# In-situ precipitation and air temperature for compound drought-heatwave monitoring

## Import libraries

In [ ]:
import math
import tempfile

import numpy as np
import xarray as xr
from c3s_eqc_automatic_quality_control import download
from scipy.stats import gamma, norm

## Define request

In [ ]:
collection_id = "insitu-gridded-observations-europe"
request = {
    "format": "zip",
    "product_type": "ensemble_mean",
    "variable": ["mean_temperature", "precipitation_amount"],
    "grid_resolution": "0_25deg",
    "period": "full_period",
    "version": "31_0e",
}

## Functions to cache

In [ ]:
def compute_mean_daylight_hours(da):
    da = da.convert_calendar("noleap", dim="time")
    dayofyear = da["time"].dt.dayofyear
    solar_declination = 0.409 * np.sin((2 * math.pi / 365) * dayofyear - 1.39)

    latitude = np.deg2rad(da["latitude"])
    sunset_hour_angle = -np.tan(latitude) * np.tan(solar_declination)
    sunset_hour_angle = xr.where(sunset_hour_angle < -1, -1, sunset_hour_angle)
    sunset_hour_angle = xr.where(sunset_hour_angle > 1, 1, sunset_hour_angle)

    daylight_hours = (24 / math.pi) * np.acos(sunset_hour_angle)
    daylight_hours = daylight_hours.resample(time="1MS").mean()
    return daylight_hours.convert_calendar("gregorian", dim="time")


def compute_monthly_pet(da):
    mean_monthly_temps = da.resample(time="1MS").mean()
    mean_monthly_temps = xr.where(mean_monthly_temps < 0, 0, mean_monthly_temps)
    heat_index = np.power(mean_monthly_temps / 5, 1.514).sum("time", skipna=False)
    a = (
        (6.75e-07 * heat_index**3)
        - (7.71e-05 * heat_index**2)
        + (1.792e-02 * heat_index)
        + 0.49239
    )

    mean_daylight_hours = da.groupby("time.year").map(compute_mean_daylight_hours)
    days_in_month = mean_daylight_hours["time"].dt.days_in_month
    days_in_month = xr.where(days_in_month == 29, 28, days_in_month)
    adjust = (mean_daylight_hours / 12) * (days_in_month / 30)
    return 16 * adjust * ((10 * mean_monthly_temps / heat_index) ** a)


def apply_thornthwaite(ds):
    return ds["tg"].groupby("time.year").map(compute_monthly_pet).fillna(0)


def _calc_spei(precip):
    count_nonzero_non_nan = np.sum((precip != 0) & ~np.isnan(precip))
    if count_nonzero_non_nan < 3:
        return np.full_like(precip, np.nan)

    precip = np.array(precip)
    shape, loc, scale = gamma.fit(precip)
    gamma_dist = gamma(shape, loc=loc, scale=scale)
    cdf = gamma_dist.cdf(precip)
    return norm.ppf(cdf)


def _add_chunksizes(da):
    da.encoding["chunksizes"] = tuple(120 if dim == "time" else 30 for dim in da.dims)
    return da


def compute_spei(ds, scale):
    pet = apply_thornthwaite(ds)
    precip = ds["rr"].resample(time="1MS").sum()
    p_minus_pet = (precip - pet).chunk(time=-1, latitude=30, longitude=30)
    scaled = p_minus_pet.rolling(time=scale, min_periods=1).sum()

    spei = xr.apply_ufunc(
        _calc_spei,
        scaled,
        input_core_dims=[["time"]],
        output_core_dims=[["time"]],
        vectorize=True,
        dask="parallelized",
        dask_gufunc_kwargs={"allow_rechunk": True},
        output_dtypes=[float],
        keep_attrs=True,
    )
    spei = spei.where(np.isfinite(spei))
    spei = spei.drop_attrs()
    return _add_chunksizes(spei).to_dataset(name="spei")


def compute_heatwaves(ds, time_slice, window, n_days):
    pad = window // 2
    da = ds["tg"]

    clim = (
        da.sel(time=time_slice)
        .groupby("time.dayofyear")
        .quantile(0.9, dim="time", method="nearest")
        .pad(dayofyear=pad, mode="wrap")
        .rolling(dayofyear=window, center=True)
        .mean()
        .isel(dayofyear=slice(pad, -pad))
    )

    hw = da.groupby("time.dayofyear") > clim
    with tempfile.TemporaryDirectory(suffix=".zarr") as tmpzarr:
        hw.chunk(time=-1, latitude=30, longitude=30).to_zarr(tmpzarr)
        hw = xr.open_dataarray(tmpzarr, chunks={})
        hw = hw.rolling(time=n_days).sum() == n_days
        hw = hw.resample(time="MS").sum()
        hw = spei.drop_attrs()
        return _add_chunksizes(hw).to_dataset(name="hw").compute()

## SPEI

In [ ]:
spei = download.download_and_transform(
    collection_id,
    request,
    transform_func=compute_spei,
    transform_func_kwargs={"scale": 3},
)["spei"]

## Heatwaves

In [ ]:
hw = download.download_and_transform(
    collection_id,
    request,
    transform_func=compute_heatwaves,
    transform_func_kwargs={
        "time_slice": slice("1961-01-01", "1990-12-31"),
        "window": 15,
        "n_days": 3,
    },
)["hw"]